In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pathlib import Path
%matplotlib inline


In [ ]:
path_to_neuroeval = Path("${COCHDNN_FMRI_RESULTS_DIR}/rsa_analysis_audio_ssl")

results_fname = "AUDIO_SSL_MODELS_CCN2026_best_layer_rsa_analysis_dict.pckl"

all_data_fname = "AUDIO_SSL_MODELS_CCN2026_all_dataset_rsa.pckl"

rsa_analysis_dict = pd.read_pickle(path_to_neuroeval / results_fname)

all_data = pd.read_pickle(path_to_neuroeval / all_data_fname)

In [ ]:
# Follow the data loading pattern from plot_utils_AUD_RSA.py
# roi=None -> all_dataset_rsa.pckl (no ROI breakdown)
# roi!=None -> best_layer_rsa_analysis_dict.pckl (with ROI)

rsa_all_rois = rsa_analysis_dict["rsa_analysis_dict_all_rois"]
N_SPLITS = 10

DATASETS = ("NH2015", "B2021")
VALUE_OF_INTEREST = "all_data_rsa_for_best_layer"


In [ ]:
# Build tidy DataFrame following plot_utils_AUD_RSA.py loading convention
# Includes both:
#   - roi-specific rows from best_layer pickle (rsa_analysis_dict)
#   - roi=None rows from all_dataset pickle (all_data)

rows = []

# --- ROI-specific data (from best_layer pickle) ---
# Navigation: rsa_data_dictionary['rsa_analysis_dict_all_rois'][target][roi][model][participant]
for dataset in DATASETS:
    if dataset not in rsa_all_rois:
        continue
    for roi, roi_dict in rsa_all_rois[dataset].items():
        for model, part_dict in roi_dict.items():
            for participant_id, pdata in part_dict.items():
                if not str(participant_id).startswith("participant"):
                    continue
                layers = pdata["model_layers"]
                values = np.asarray(pdata[VALUE_OF_INTEREST], dtype=float)
                assert len(layers) == len(values), (
                    f"{dataset}/{roi}/{model}/{participant_id}: "
                    f"len(model_layers)={len(layers)} != len({VALUE_OF_INTEREST})={len(values)}"
                )
                for layer_name, v in zip(layers, values):
                    rows.append({
                        "dataset": dataset,
                        "roi": roi,
                        "model": model,
                        "participant": participant_id,
                        "layer_name": layer_name,
                        "rsa_value": float(v),
                        "value_of_interest": VALUE_OF_INTEREST,
                    })

# --- Whole-brain / no ROI data (from all_dataset pickle) ---
# Navigation: rsa_data_dictionary[target]['trained'][model][participant]
for dataset in DATASETS:
    if dataset not in all_data:
        continue
    trained = all_data[dataset].get("trained", {})
    for model, part_dict in trained.items():
        for participant_id, pdata in part_dict.items():
            if not str(participant_id).startswith("participant"):
                continue
            layers = pdata["model_layers"]
            values = np.asarray(pdata[VALUE_OF_INTEREST], dtype=float)
            assert len(layers) == len(values), (
                f"{dataset}/none/{model}/{participant_id}: "
                f"len(model_layers)={len(layers)} != len({VALUE_OF_INTEREST})={len(values)}"
            )
            for layer_name, v in zip(layers, values):
                rows.append({
                    "dataset": dataset,
                    "roi": "none",
                    "model": model,
                    "participant": participant_id,
                    "layer_name": layer_name,
                    "rsa_value": float(v),
                    "value_of_interest": VALUE_OF_INTEREST,
                })

df_median_layer_rsa = pd.DataFrame(rows)
print(f"Shape: {df_median_layer_rsa.shape}")
print(f"Datasets: {df_median_layer_rsa['dataset'].unique()}")
print(f"ROIs: {df_median_layer_rsa['roi'].unique()}")
df_median_layer_rsa.groupby(["dataset", "roi", "model"], sort=False).size().head(20)


In [ ]:
# Spot-check: stored per-layer value matches what plot_utils_AUD_RSA would load
dataset = "NH2015"
if dataset not in rsa_all_rois:
    dataset = next(iter(rsa_all_rois.keys()))
roi = next(iter(rsa_all_rois[dataset].keys()))
model = next(iter(rsa_all_rois[dataset][roi].keys()))
part = next(k for k in rsa_all_rois[dataset][roi][model] if str(k).startswith("participant"))
pdata = rsa_all_rois[dataset][roi][model][part]
layers = pdata["model_layers"]
j = 0
stored = float(np.asarray(pdata[VALUE_OF_INTEREST], dtype=float)[j])
layer_name = layers[j]
from_df = df_median_layer_rsa.loc[
    (df_median_layer_rsa["dataset"] == dataset)
    & (df_median_layer_rsa["roi"] == roi)
    & (df_median_layer_rsa["model"] == model)
    & (df_median_layer_rsa["participant"] == part)
    & (df_median_layer_rsa["layer_name"] == layer_name),
    "rsa_value",
].iloc[0]
np.testing.assert_allclose(stored, from_df, rtol=0, atol=1e-12)
print(f"OK: {dataset} {roi} {model} {part} layer={layer_name!r} value={stored:.6f}")


In [ ]:
import sys
from pathlib import Path

_REPO = Path.cwd().resolve()
if not (_REPO / "figure_utils.py").exists():
    _REPO = _REPO.parent
sys.path.insert(0, str(_REPO))

import figure_utils
from importlib import reload
reload(figure_utils)
from figure_utils import (
    build_model_palette,
    format_model_str,
    get_standard_base_colors,
    get_standard_hue_order,
    normalize_model_name,
    model_label,
)


def rsa_key_to_model_plot(key: str) -> str:
    if key == "spectemp":
        return "spectemp"
    try:
        name, _, _, _ = format_model_str(key)
        return name
    except Exception:
        return normalize_model_name(str(key).replace("_", " "))


PLOT_DATASET = "NH2015"

plot_base = df_median_layer_rsa[(df_median_layer_rsa["dataset"] == PLOT_DATASET) & (df_median_layer_rsa['roi'] != 'none')].copy()
plot_base["model_plot"] = plot_base["model"].map(rsa_key_to_model_plot)

# Drop "unknown supervised" and spectemp from lines
drop_names = {"unknown supervised"}
spectemp_names = set(
    plot_base.loc[plot_base["model"].str.lower() == "spectemp", "model_plot"].unique()
)
if not spectemp_names:
    spectemp_names = {"spectemp"}

# Compute spectemp mean per ROI per layer for hlines
spectemp_df = plot_base[plot_base["model_plot"].isin(spectemp_names)]
spectemp_mean = spectemp_df.groupby("roi")["rsa_value"].mean()

# Remove spectemp + unknown from line data
to_plot = plot_base[~plot_base["model_plot"].isin(drop_names | spectemp_names)].copy()

_ref_model = to_plot["model"].iloc[0]
layer_order = (
    to_plot.loc[to_plot["model"] == _ref_model, "layer_name"]
    .drop_duplicates()
    .tolist()
)
for lyr in to_plot["layer_name"].unique():
    if lyr not in layer_order:
        layer_order.append(lyr)

std_order = get_standard_hue_order()
present = sorted(to_plot["model_plot"].unique())
hue_order = [m for m in std_order if m in present] + [m for m in present if m not in std_order]
to_plot = to_plot[to_plot["model_plot"].isin(hue_order)].copy()

base_colors = get_standard_base_colors()
hue_dict = build_model_palette(hue_order, base_colors)

roi_order = sorted(to_plot["roi"].unique())
n_rois = len(roi_order)

panel_size = 3.5
title_fontsize = 10

n_cols = 2
n_rows = int(np.ceil(n_rois / n_cols))
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(panel_size * n_cols, panel_size * n_rows),
    sharey=True,
)
axes = np.atleast_1d(axes).flatten()

for i, roi in enumerate(roi_order):
    ax = axes[i]
    roi_data = to_plot[to_plot["roi"] == roi]

    sns.pointplot(
        data=roi_data,
        x="layer_name",
        y="rsa_value",
        hue="model_plot",
        hue_order=hue_order,
        palette=hue_dict,
        order=layer_order,
        ax=ax,
        markers="o",
        dodge=True,
        markersize=4,
        linestyle="-",
        linewidth=1,
        errorbar="se",
    )
    for line in ax.lines:
        line.set_markeredgecolor("black")
        line.set_markeredgewidth(0.5)

    if roi in spectemp_mean.index:
        ax.axhline(
            spectemp_mean[roi], color="k", linestyle="--", linewidth=1.5,
            zorder=5, label="spectemp",
        )

    ax.set_title(roi, fontsize=title_fontsize)
    ax.set_xlabel("")
    if i >= n_rois - n_cols:
        ax.set_xlabel("Layer", fontsize=title_fontsize)
    ax.set_ylabel("")
    if i % n_cols == 0:
        ax.set_ylabel("RSA (all sounds)", fontsize=title_fontsize)
    ax.set_aspect("auto")
    ax.set_box_aspect(1)

    ax.legend().remove()
    ax.tick_params(axis="x", labelsize=title_fontsize)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.tick_params(axis="y", labelsize=title_fontsize)
    sns.despine(ax=ax)

for j in range(n_rois, len(axes)):
    axes[j].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
labels = [model_label(l, use_ssl_names=True) for l in labels]
fig.legend(
    handles,
    labels,
    loc="center right",
    bbox_to_anchor=(0.85, 0.97),
    fontsize=8,
    frameon=False,
    ncol=min(4, len(labels)),
)
fig.suptitle(
    f"{PLOT_DATASET} — layer-wise RSA (spectemp = dashed line)",
    y=1.05,
    fontsize=title_fontsize,
)

plt.tight_layout()


In [ ]:
# Same models as figure_4_plot_fmri_components.ipynb main plot
FIGURE4_MODELS = [
    "kell2018_word_speaker_audioset_MatchedDataset_LARS_latest_ckpt",
    "word_kell2018_MatchedDataset_LARS",
    "audioset_kell2018_MatchedDataset_LARS_latest_ckpt",
    "kell2018_audioset_unbalanced_supervised",
    "kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_0e-01",
    "kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01_latest_ckpt",
    "kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_0e-01_audioset_only",
    "kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01_audioset_only",
    "spectemp",
]
FIG4_EXCLUDE = "0.1|0.2|0.3|0.4|1.0|dual"

PLOT_DATASET = "NH2015"

fig4_base = df_median_layer_rsa[
    (df_median_layer_rsa["dataset"] == PLOT_DATASET)
    & (df_median_layer_rsa["model"].isin(FIGURE4_MODELS))
    & (df_median_layer_rsa["roi"] != "none")
].copy()
fig4_base["model_plot"] = fig4_base["model"].map(rsa_key_to_model_plot)
fig4_base = fig4_base[~fig4_base["model_plot"].str.contains(FIG4_EXCLUDE, na=False)]

spectemp_names_f4 = set(
    fig4_base.loc[fig4_base["model"].str.lower() == "spectemp", "model_plot"].unique()
)
if not spectemp_names_f4:
    spectemp_names_f4 = {"spectemp"}
spectemp_mean_f4 = (
    fig4_base[fig4_base["model_plot"].isin(spectemp_names_f4)]
    .groupby("roi")["rsa_value"]
    .mean()
)
to_plot_f4 = fig4_base[~fig4_base["model_plot"].isin(spectemp_names_f4)].copy()

_ref_f4 = to_plot_f4["model"].iloc[0]
layer_order_f4 = (
    to_plot_f4.loc[to_plot_f4["model"] == _ref_f4, "layer_name"]
    .drop_duplicates()
    .tolist()
)
for lyr in to_plot_f4["layer_name"].unique():
    if lyr not in layer_order_f4:
        layer_order_f4.append(lyr)

std_order = get_standard_hue_order()
present_f4 = sorted(to_plot_f4["model_plot"].unique())
hue_order_f4 = [m for m in std_order if m in present_f4] + [
    m for m in present_f4 if m not in std_order
]
to_plot_f4 = to_plot_f4[to_plot_f4["model_plot"].isin(hue_order_f4)].copy()

base_colors = get_standard_base_colors()
hue_dict_f4 = build_model_palette(hue_order_f4, base_colors)

roi_order_f4 = sorted(to_plot_f4["roi"].unique())
n_rois_f4 = len(roi_order_f4)
n_cols = 2
n_rows = int(np.ceil(n_rois_f4 / n_cols))

panel_size = 3.5
title_fontsize = 10

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(panel_size * n_cols, panel_size * n_rows),
    sharey=True,
)
axes = np.atleast_1d(axes).flatten()

for i, roi in enumerate(roi_order_f4):
    ax = axes[i]
    roi_data = to_plot_f4[to_plot_f4["roi"] == roi]

    sns.pointplot(
        data=roi_data,
        x="layer_name",
        y="rsa_value",
        hue="model_plot",
        hue_order=hue_order_f4,
        palette=hue_dict_f4,
        order=layer_order_f4,
        ax=ax,
        markers="o",
        dodge=True,
        markersize=4,
        linestyle="-",
        linewidth=1,
        errorbar="se",
    )
    for line in ax.lines:
        line.set_markeredgecolor("black")
        line.set_markeredgewidth(0.5)

    if roi in spectemp_mean_f4.index:
        ax.axhline(
            spectemp_mean_f4[roi], color="k", linestyle="--", linewidth=1.5,
            zorder=5, label="spectemp",
        )

    ax.set_title(roi, fontsize=title_fontsize)
    ax.set_xlabel("")
    if i >= n_rois_f4 - n_cols:
        ax.set_xlabel("Layer", fontsize=title_fontsize)
    ax.set_ylabel("")
    if i % n_cols == 0:
        ax.set_ylabel("RSA (all sounds)", fontsize=title_fontsize)
    ax.set_aspect("auto")
    ax.set_box_aspect(1)

    ax.legend().remove()
    ax.tick_params(axis="x", labelsize=title_fontsize)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.tick_params(axis="y", labelsize=title_fontsize)
    sns.despine(ax=ax)

for j in range(n_rois_f4, len(axes)):
    axes[j].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
labels = [model_label(l, use_ssl_names=True) for l in labels]
fig.legend(
    handles,
    labels,
    loc="center right",
    bbox_to_anchor=(0.85, 0.97),
    fontsize=8,
    frameon=False,
    ncol=min(4, len(labels)),
)
fig.suptitle(
    f"{PLOT_DATASET} — layer-wise RSA (figure 4 models, spectemp = dashed)",
    y=1.05,
    fontsize=title_fontsize,
)



In [ ]:
# ---------------------------------------------------------------------------
# Load CochCNN9 layerwise zero-shot evaluation results
# ---------------------------------------------------------------------------
RESULTS_DIR = Path("../results_dfs")
METRIC_COL = "sqr_l2_judgement"

TASK_INFO = {
    "speech_commands": "Loudness order discrimination",
    "nsynth": "Melody match",
    "mandarin": "Mandarin tone discrimination",
}

zs_frames = []
for task_key, task_label in TASK_INFO.items():
    fpath = RESULTS_DIR / f"zero_shot_{task_key}_cochdnn_layerwise.csv"
    df_t = pd.read_csv(fpath)
    df_t["task"] = task_label
    zs_frames.append(df_t)

df_zs_raw = pd.concat(zs_frames, ignore_index=True)

# Layer order in network depth (from RSA data, which stores model_layers in depth order)
_ref_rsa_model = df_median_layer_rsa["model"].iloc[0]
RSA_LAYERS = (
    df_median_layer_rsa
    .loc[df_median_layer_rsa["model"] == _ref_rsa_model, "layer_name"]
    .drop_duplicates()
    .tolist()
)
df_zs_raw = df_zs_raw[df_zs_raw["layer"].isin(RSA_LAYERS)].copy()
df_zs_raw["layer"] = pd.Categorical(df_zs_raw["layer"], categories=RSA_LAYERS, ordered=True)

# Aggregate: mean accuracy per (model_name, layer, task)
df_zs = (
    df_zs_raw
    .groupby(["model_name", "layer", "task"], as_index=False, observed=True)[METRIC_COL]
    .mean()
    .rename(columns={METRIC_COL: "accuracy"})
)

# Unify the one naming mismatch with the RSA data
ZS_TO_RSA = {"CochCNN9 scaled supervised": "CochCNN9 scaled aud. event"}
df_zs["model_plot"] = df_zs["model_name"].replace(ZS_TO_RSA)

print(f"df_zs: {df_zs.shape[0]} rows")
print(f"  tasks:  {sorted(df_zs['task'].unique())}")
print(f"  models: {sorted(df_zs['model_plot'].unique())}")
print(f"  layers: {sorted(df_zs['layer'].unique())}")
df_zs.head()

In [ ]:
# ---------------------------------------------------------------------------
# Plot 1: Zero-shot task performance by layer  (same format as fMRI RSA plots)
# ---------------------------------------------------------------------------
task_order = sorted(df_zs["task"].unique())

_ref_zs = df_zs["model_plot"].iloc[0]
layer_order_zs = (
    df_zs.loc[df_zs["model_plot"] == _ref_zs, "layer"]
    .drop_duplicates()
    .tolist()
)
for lyr in df_zs["layer"].unique():
    if lyr not in layer_order_zs:
        layer_order_zs.append(lyr)

# Reuse figure_utils palette
std_order = get_standard_hue_order()
present_zs = sorted(df_zs["model_plot"].unique())
hue_order_zs = [m for m in std_order if m in present_zs] + [
    m for m in present_zs if m not in std_order
]
base_colors = get_standard_base_colors()
hue_dict_zs = build_model_palette(hue_order_zs, base_colors)

n_tasks = len(task_order)
n_cols_zs = min(n_tasks, 3)
n_rows_zs = int(np.ceil(n_tasks / n_cols_zs))
panel_size = 3.5
title_fontsize = 10

fig_zs, axes_zs = plt.subplots(
    n_rows_zs, n_cols_zs,
    figsize=(panel_size * n_cols_zs, panel_size * n_rows_zs),
    sharey=True,
    squeeze=False,
)
axes_zs = axes_zs.flatten()

for i, task in enumerate(task_order):
    ax = axes_zs[i]
    task_data = df_zs[df_zs["task"] == task]

    sns.pointplot(
        data=task_data,
        x="layer",
        y="accuracy",
        hue="model_plot",
        hue_order=hue_order_zs,
        palette=hue_dict_zs,
        order=layer_order_zs,
        ax=ax,
        markers="o",
        dodge=True,
        markersize=4,
        linestyle="-",
        linewidth=1,
    )
    for line in ax.lines:
        line.set_markeredgecolor("black")
        line.set_markeredgewidth(0.5)

    ax.set_title(task, fontsize=title_fontsize)
    ax.set_xlabel("")
    if i >= n_tasks - n_cols_zs:
        ax.set_xlabel("Layer", fontsize=title_fontsize)
    ax.set_ylabel("")
    if i % n_cols_zs == 0:
        ax.set_ylabel(f"Zero-shot accuracy\n(sqr. L2 judgement)", fontsize=title_fontsize)
    ax.set_aspect("auto")
    ax.set_box_aspect(1)

    ax.legend().remove()
    ax.tick_params(axis="x", labelsize=8)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.tick_params(axis="y", labelsize=title_fontsize)
    ax.set_ylim(0.5, 1.0)
    sns.despine(ax=ax)

for j in range(n_tasks, len(axes_zs)):
    axes_zs[j].set_visible(False)

handles, labels = axes_zs[0].get_legend_handles_labels()
labels = [model_label(l, use_ssl_names=True) for l in labels]
fig_zs.legend(
    handles, labels,
    loc="center right",
    bbox_to_anchor=(0.85, 0.97),
    fontsize=8,
    frameon=False,
    ncol=min(4, len(labels)),
)
fig_zs.suptitle("CochCNN9 — layer-wise zero-shot task performance", y=1.05, fontsize=title_fontsize)
plt.tight_layout()

In [ ]:
# ---------------------------------------------------------------------------
# Zero-shot performance at the best-RSA layer (per model, per ROI)
# ---------------------------------------------------------------------------
PLOT_DATASET_BEST = "NH2015"

# Find best RSA layer per (model, roi): highest mean RSA across participants
_rsa_filt_best = df_median_layer_rsa[
    (df_median_layer_rsa["dataset"] == PLOT_DATASET_BEST)
    & (df_median_layer_rsa["roi"] != "none")
    & (df_median_layer_rsa["model"] != "spectemp")
].copy()
_rsa_filt_best["model_plot"] = _rsa_filt_best["model"].map(rsa_key_to_model_plot)
rsa_mean = (
    _rsa_filt_best
    .groupby(["model_plot", "layer_name", "roi"], as_index=False)["rsa_value"]
    .mean()
)
best_idx = rsa_mean.groupby(["model_plot", "roi"])["rsa_value"].idxmax()
best_layers = rsa_mean.loc[best_idx, ["model_plot", "roi", "layer_name"]].copy()

# Merge with zero-shot to get accuracy at those best layers
zs_at_best = pd.merge(
    best_layers,
    df_zs,
    left_on=["model_plot", "layer_name"],
    right_on=["model_plot", "layer"],
    how="inner",
)
print(f"Zero-shot at best RSA layers: {zs_at_best.shape[0]} rows")
print(f"  models: {sorted(zs_at_best['model_plot'].unique())}")

# Pretty label
zs_at_best["model_lbl"] = zs_at_best["model_plot"].map(
    lambda m: model_label(m, use_ssl_names=True)
)

# Order models by standard palette
present_best = sorted(zs_at_best["model_plot"].unique())
hue_order_best = [m for m in std_order if m in present_best] + [
    m for m in present_best if m not in std_order
]
label_order_best = [model_label(m, use_ssl_names=True) for m in hue_order_best]
palette_best = {
    model_label(m, use_ssl_names=True): build_model_palette([m], base_colors)[m]
    for m in hue_order_best
}

task_list = sorted(zs_at_best["task"].unique())
roi_list = sorted(zs_at_best["roi"].unique())
n_rois_b = len(roi_list)

fig_best, axes_best = plt.subplots(
    1, n_rois_b,
    figsize=(4.5 * n_rois_b, 4),
    sharey=True,
)
if n_rois_b == 1:
    axes_best = [axes_best]

for ri, roi in enumerate(roi_list):
    ax = axes_best[ri]
    rdata = zs_at_best[zs_at_best["roi"] == roi]

    sns.barplot(
        data=rdata,
        x="task",
        y="accuracy",
        hue="model_lbl",
        hue_order=label_order_best,
        palette=palette_best,
        order=task_list,
        ax=ax,
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_ylim(0.5, 1.0)
    ax.set_title(roi, fontsize=10)
    ax.set_xlabel("Task", fontsize=9)
    ax.set_ylabel("")
    if ri == 0:
        ax.set_ylabel("Zero-shot accuracy\n(at best RSA layer)", fontsize=9)
    ax.tick_params(axis="both", labelsize=8)
    ax.tick_params(axis="x", rotation=25)
    ax.legend().remove()
    sns.despine(ax=ax)

handles, labels = axes_best[0].get_legend_handles_labels()
fig_best.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.12),
    fontsize=7,
    frameon=False,
    ncol=min(4, len(labels)),
)
fig_best.suptitle(
    f"{PLOT_DATASET_BEST} — Zero-shot at best RSA layer (per model × ROI)",
    y=1.18, fontsize=11,
)
plt.tight_layout()

In [ ]:
# ---------------------------------------------------------------------------
# Plot 2: Correlate zero-shot accuracy with fMRI RSA  (lmplot style)
#   x = RSA score (mean across participants), y = zero-shot accuracy
#   Each point = one (model, layer).  sns.lmplot draws regression + CI.
#   Facet: columns = ROI, rows = task
# ---------------------------------------------------------------------------
from scipy import stats

PLOT_DATASET_CORR = "NH2015"
EXCLUDE_ROI = {"none"}

# Aggregate RSA to mean across participants; group by model_plot to avoid duplicates
_rsa_filt = df_median_layer_rsa[
    (df_median_layer_rsa["dataset"] == PLOT_DATASET_CORR)
    & (~df_median_layer_rsa["roi"].isin(EXCLUDE_ROI))
    & (df_median_layer_rsa["model"] != "spectemp")
].copy()
_rsa_filt["model_plot"] = _rsa_filt["model"].map(rsa_key_to_model_plot)
rsa_for_merge = (
    _rsa_filt
    .groupby(["model_plot", "layer_name", "roi"], as_index=False)["rsa_value"]
    .mean()
)

# Merge: zero-shot <-> RSA on (model_plot, layer)
merged = pd.merge(
    df_zs,
    rsa_for_merge,
    left_on=["model_plot", "layer"],
    right_on=["model_plot", "layer_name"],
    how="inner",
)

# Pretty labels for the legend
merged["model_label"] = merged["model_plot"].map(
    lambda m: model_label(m, use_ssl_names=True)
)

print(f"Merged rows: {merged.shape[0]}")
print(f"  models:  {sorted(merged['model_plot'].unique())}")
print(f"  tasks:   {sorted(merged['task'].unique())}")
print(f"  ROIs:    {sorted(merged['roi'].unique())}")

# Build palette keyed by model_label (what lmplot will see as hue)
present_corr = sorted(merged["model_plot"].unique())
hue_order_corr = [m for m in std_order if m in present_corr] + [
    m for m in present_corr if m not in std_order
]
label_order = [model_label(m, use_ssl_names=True) for m in hue_order_corr]
label_palette = {
    model_label(m, use_ssl_names=True): build_model_palette([m], base_colors)[m]
    for m in hue_order_corr
}

g = sns.lmplot(
    data=merged,
    x="rsa_value",
    y="accuracy",
    hue="model_label",
    hue_order=label_order,
    palette=label_palette,
    col="roi",
    row="task",
    col_order=sorted(merged["roi"].unique()),
    row_order=sorted(merged["task"].unique()),
    height=3.2,
    aspect=1,
    scatter_kws=dict(s=22, alpha=0.6, edgecolors="black", linewidths=0.3),
    line_kws=dict(linewidth=2),
    ci=95,
    facet_kws=dict(sharex="col", sharey="row", margin_titles=True),
)

g.set_axis_labels("RSA (all sounds)", "Zero-shot accuracy", fontsize=9)
g.set_titles(row_template="{row_name}", col_template="{col_name}", size=10)

for ax in g.axes.flat:
    ax.tick_params(axis="both", labelsize=8)
    ax.set_ylim(0.5, 1.0)
    sns.despine(ax=ax)

# Annotate each panel with overall Pearson r
for ti, task in enumerate(sorted(merged["task"].unique())):
    for ri, roi in enumerate(sorted(merged["roi"].unique())):
        ax = g.axes[ti, ri]
        sub = merged[(merged["roi"] == roi) & (merged["task"] == task)]
        if len(sub) >= 5:
            r_val, p_val = stats.pearsonr(sub["rsa_value"], sub["accuracy"])
            ax.annotate(
                f"r={r_val:.2f}, p={p_val:.3f}",
                xy=(0.05, 0.95), xycoords="axes fraction",
                fontsize=7, va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
            )

g.figure.subplots_adjust(top=0.92)
g.figure.suptitle(
    f"{PLOT_DATASET_CORR} — Zero-shot accuracy vs. fMRI RSA (per layer)",
    fontsize=11,
)


In [ ]:
# ---------------------------------------------------------------------------
# Plot 3: Zero-shot vs RSA — rows = models, cols = tasks, hue = ROI
# ---------------------------------------------------------------------------

ROI_PALETTE = {
    "Anterior": "#e41a1c",
    "Lateral": "#377eb8",
    "Posterior": "#4daf4a",
    "Primary": "#984ea3",
}

# Use model_label for nice row names
merged["model_row"] = merged["model_plot"].map(
    lambda m: model_label(m, use_ssl_names=True)
)

# Ordered by standard palette order
model_row_order = [
    model_label(m, use_ssl_names=True)
    for m in hue_order_corr
    if model_label(m, use_ssl_names=True) in merged["model_row"].values
]

roi_order_plot = [r for r in ["Primary", "Lateral", "Posterior", "Anterior"]
                  if r in merged["roi"].values]
roi_pal = {r: ROI_PALETTE[r] for r in roi_order_plot}

g2 = sns.lmplot(
    data=merged,
    x="rsa_value",
    y="accuracy",
    hue="roi",
    hue_order=roi_order_plot,
    palette=roi_pal,
    row="model_row",
    col="task",
    row_order=model_row_order,
    col_order=sorted(merged["task"].unique()),
    height=2.5,
    aspect=1.1,
    scatter_kws=dict(s=22, alpha=0.6, edgecolors="black", linewidths=0.3),
    line_kws=dict(linewidth=2),
    ci=95,
    facet_kws=dict(sharex=False, sharey=False, margin_titles=False),
)

g2.set_axis_labels("RSA (all sounds)", "Zero-shot accuracy", fontsize=9)
g2.set_titles(template="{row_name} \n {col_name}", size=8)

for ax in g2.axes.flat:
    ax.tick_params(axis="both", labelsize=7)
    ax.set_ylim(0.5, 1.0)
    ax.set_xlim(merged["rsa_value"].min() - 0.01, merged["rsa_value"].max() + 0.01)
    sns.despine(ax=ax)

# Annotate each panel with Pearson r
for ri, mrow in enumerate(model_row_order):
    for ci, task in enumerate(sorted(merged["task"].unique())):
        ax = g2.axes[ri, ci]
        sub = merged[(merged["model_row"] == mrow) & (merged["task"] == task)]
        if len(sub) >= 5:
            r_val, p_val = stats.pearsonr(sub["rsa_value"], sub["accuracy"])
            ax.annotate(
                f"r={r_val:.2f}",
                xy=(0.95, 0.05), xycoords="axes fraction",
                fontsize=7, va="bottom", ha="right",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
            )

g2.figure.subplots_adjust(top=0.94)
g2.figure.suptitle(
    f"{PLOT_DATASET_CORR} — Zero-shot vs RSA by model (lines = ROI)",
    fontsize=11,
)

In [ ]:
# ---------------------------------------------------------------------------
# Load figure_4 fMRI component data (median R² across layers)
# ---------------------------------------------------------------------------
RESULTDIR_ROOT = Path("${COCHDNN_FMRI_RESULTS_DIR}")

FIGURE4_MODELS_RAW = [
    'kell2018_word_speaker_audioset_MatchedDataset_LARS_latest_ckpt',
    'word_kell2018_MatchedDataset_LARS',
    'audioset_kell2018_MatchedDataset_LARS_latest_ckpt',
    'kell2018_audioset_unbalanced_supervised',
    'kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_0e-01',
    'kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_1e-01_latest_ckpt',
    'kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_2e-01_latest_ckpt',
    'kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_3e-01_latest_ckpt',
    'kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_4e-01_latest_ckpt',
    'kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01_latest_ckpt',
    'kell2018_dual_barlow_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01',
    'kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_0e-01_audioset_only',
    'kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01_audioset_only',
]

COMP_NAMES = {
    'lowfreq': 'Low-freq',
    'highfreq': 'High-freq',
    'envsounds': 'Env. sounds',
    'pitch': 'Pitch',
    'speech': 'Speech',
    'music': 'Music',
}

comp_rows = []
for source_model in FIGURE4_MODELS_RAW:
    outputs_dir = RESULTDIR_ROOT / source_model / 'outputs'
    for comp_key, comp_label in COMP_NAMES.items():
        csv_path = outputs_dir / f'across-layers_comp-{comp_key}_{source_model}_NH2015comp_median_r2_test.csv'
        if not csv_path.exists():
            continue
        df_comp = pd.read_csv(csv_path, index_col=0)
        if 'none' not in df_comp.index:
            continue
        scores = df_comp.loc['none']
        for layer in scores.index:
            comp_rows.append(dict(
                model_raw=source_model,
                layer=layer,
                component=comp_label,
                comp_key=comp_key,
                score=float(scores[layer]),
            ))

df_comp_all = pd.DataFrame(comp_rows)
df_comp_all["model_plot"] = df_comp_all["model_raw"].map(
    lambda m: format_model_str(m)[0] if m != "spectemp" else "spectemp"
)

print(f"df_comp_all: {df_comp_all.shape[0]} rows")
print(f"  models:     {sorted(df_comp_all['model_plot'].unique())}")
print(f"  components: {sorted(df_comp_all['component'].unique())}")
print(f"  layers:     {sorted(df_comp_all['layer'].unique())}")
df_comp_all.head()

In [ ]:
# ---------------------------------------------------------------------------
# Plot: fMRI component R² by layer (same format as zero-shot / RSA plots)
#   — excludes intermediate lambdas and dual, matching the figure_4 main plot
# ---------------------------------------------------------------------------
FIG4_EXCLUDE = "0.1|0.2|0.3|0.4|1.0|dual"
comp_plot = df_comp_all[~df_comp_all["model_plot"].str.contains(FIG4_EXCLUDE, na=False)].copy()

_ref_comp = comp_plot["model_raw"].iloc[0]
layer_order_comp = comp_plot.loc[comp_plot["model_raw"] == _ref_comp, "layer"].drop_duplicates().tolist()
for lyr in comp_plot["layer"].unique():
    if lyr not in layer_order_comp:
        layer_order_comp.append(lyr)

present_comp = sorted(comp_plot["model_plot"].unique())
hue_order_comp = [m for m in std_order if m in present_comp] + [
    m for m in present_comp if m not in std_order
]
hue_dict_comp = build_model_palette(hue_order_comp, base_colors)

comp_list = list(COMP_NAMES.values())
n_comp = len(comp_list)
n_cols_comp = 3
n_rows_comp = int(np.ceil(n_comp / n_cols_comp))
panel_size = 3.5
title_fontsize = 10

fig_comp, axes_comp = plt.subplots(
    n_rows_comp, n_cols_comp,
    figsize=(panel_size * n_cols_comp, panel_size * n_rows_comp),
    sharey=True,
    squeeze=False,
)
axes_comp = axes_comp.flatten()

for i, comp in enumerate(comp_list):
    ax = axes_comp[i]
    cdata = comp_plot[comp_plot["component"] == comp]

    sns.pointplot(
        data=cdata,
        x="layer",
        y="score",
        hue="model_plot",
        hue_order=hue_order_comp,
        palette=hue_dict_comp,
        order=layer_order_comp,
        ax=ax,
        markers="o",
        dodge=True,
        markersize=4,
        linestyle="-",
        linewidth=1,
    )
    for line in ax.lines:
        line.set_markeredgecolor("black")
        line.set_markeredgewidth(0.5)

    ax.set_title(comp, fontsize=title_fontsize)
    ax.set_xlabel("")
    if i >= n_comp - n_cols_comp:
        ax.set_xlabel("Layer", fontsize=title_fontsize)
    ax.set_ylabel("")
    if i % n_cols_comp == 0:
        ax.set_ylabel("Median $R^2$", fontsize=title_fontsize)
    ax.set_aspect("auto")
    ax.set_box_aspect(1)

    ax.legend().remove()
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.tick_params(axis="y", labelsize=title_fontsize)
    sns.despine(ax=ax)

for j in range(n_comp, len(axes_comp)):
    axes_comp[j].set_visible(False)

handles, labels = axes_comp[0].get_legend_handles_labels()
labels = [model_label(l, use_ssl_names=True) for l in labels]
fig_comp.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.08),
    fontsize=8,
    frameon=False,
    ncol=min(4, len(labels)),
)
fig_comp.suptitle("fMRI component predictions by layer (figure 4 models)", y=1.12, fontsize=11)
plt.tight_layout()

In [ ]:
# ---------------------------------------------------------------------------
# Correlate zero-shot accuracy with fMRI component R² (lmplot)
#   x = component R², y = zero-shot accuracy
#   Each point = one (model, layer). Facet: col = component, row = task
# ---------------------------------------------------------------------------

# Merge component data with zero-shot (on model_plot + layer)
merged_comp = pd.merge(
    df_zs,
    comp_plot[["model_plot", "layer", "component", "score"]],
    on=["model_plot", "layer"],
    how="inner",
)

merged_comp["model_lbl"] = merged_comp["model_plot"].map(
    lambda m: model_label(m, use_ssl_names=True)
)

print(f"Merged (ZS × component): {merged_comp.shape[0]} rows")
print(f"  models:     {sorted(merged_comp['model_plot'].unique())}")
print(f"  tasks:      {sorted(merged_comp['task'].unique())}")
print(f"  components: {sorted(merged_comp['component'].unique())}")

label_order_mc = [
    model_label(m, use_ssl_names=True) for m in hue_order_comp
    if model_label(m, use_ssl_names=True) in merged_comp["model_lbl"].values
]
label_palette_mc = {
    model_label(m, use_ssl_names=True): build_model_palette([m], base_colors)[m]
    for m in hue_order_comp
    if model_label(m, use_ssl_names=True) in merged_comp["model_lbl"].values
}

g_comp = sns.lmplot(
    data=merged_comp,
    x="score",
    y="accuracy",
    hue="model_lbl",
    hue_order=label_order_mc,
    palette=label_palette_mc,
    col="component",
    row="task",
    col_order=list(COMP_NAMES.values()),
    row_order=sorted(merged_comp["task"].unique()),
    height=2.8,
    aspect=1,
    scatter_kws=dict(s=18, alpha=0.5, edgecolors="black", linewidths=0.3),
    line_kws=dict(linewidth=2),
    ci=95,
    facet_kws=dict(sharex=False, sharey=False, margin_titles=False),
)

g_comp.set_axis_labels("Component $R^2$", "Zero-shot accuracy", fontsize=9)
g_comp.set_titles(template="{col_name} | {row_name}", size=8)

for ax in g_comp.axes.flat:
    ax.tick_params(axis="both", labelsize=7)
    ax.set_ylim(0.5, 1.0)
    sns.despine(ax=ax)

# Annotate each panel with overall Pearson r
comp_col_order = list(COMP_NAMES.values())
task_row_order = sorted(merged_comp["task"].unique())
for ti, task in enumerate(task_row_order):
    for ci, comp in enumerate(comp_col_order):
        ax = g_comp.axes[ti, ci]
        sub = merged_comp[(merged_comp["component"] == comp) & (merged_comp["task"] == task)]
        if len(sub) >= 5:
            r_val, p_val = stats.pearsonr(sub["score"], sub["accuracy"])
            ax.annotate(
                f"r={r_val:.2f}, p={p_val:.3f}",
                xy=(0.95, 0.05), xycoords="axes fraction",
                fontsize=6, va="bottom", ha="right",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
            )

g_comp.figure.subplots_adjust(top=0.93)
g_comp.figure.suptitle(
    "Zero-shot accuracy vs. fMRI component R² (per layer, figure 4 models)",
    fontsize=10,
)

In [ ]:
# ---------------------------------------------------------------------------
# Zero-shot at best component-R² layer (per model, per component)
# ---------------------------------------------------------------------------
best_comp_idx = comp_plot.groupby(["model_plot", "component"])["score"].idxmax()
best_comp_layers = comp_plot.loc[best_comp_idx, ["model_plot", "component", "layer"]].copy()

zs_at_best_comp = pd.merge(
    best_comp_layers,
    df_zs,
    on=["model_plot", "layer"],
    how="inner",
)
zs_at_best_comp["model_lbl"] = zs_at_best_comp["model_plot"].map(
    lambda m: model_label(m, use_ssl_names=True)
)

label_order_bc = [
    model_label(m, use_ssl_names=True) for m in hue_order_comp
    if model_label(m, use_ssl_names=True) in zs_at_best_comp["model_lbl"].values
]
palette_bc = {l: label_palette_mc[l] for l in label_order_bc if l in label_palette_mc}

task_list_bc = sorted(zs_at_best_comp["task"].unique())
n_tasks_bc = len(task_list_bc)

fig_bc, axes_bc = plt.subplots(
    1, n_tasks_bc,
    figsize=(4.5 * n_tasks_bc, 4),
    sharey=True,
)
if n_tasks_bc == 1:
    axes_bc = [axes_bc]

for ti, task in enumerate(task_list_bc):
    ax = axes_bc[ti]
    tdata = zs_at_best_comp[zs_at_best_comp["task"] == task]

    sns.barplot(
        data=tdata,
        x="component",
        y="accuracy",
        hue="model_lbl",
        hue_order=label_order_bc,
        palette=palette_bc,
        order=list(COMP_NAMES.values()),
        ax=ax,
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_ylim(0.5, 1.0)
    ax.set_title(task, fontsize=10)
    ax.set_xlabel("Component", fontsize=9)
    ax.set_ylabel("")
    if ti == 0:
        ax.set_ylabel("Zero-shot accuracy\n(at best component-R² layer)", fontsize=9)
    ax.tick_params(axis="both", labelsize=8)
    ax.legend().remove()
    sns.despine(ax=ax)

handles, labels = axes_bc[0].get_legend_handles_labels()
fig_bc.legend(
    handles, labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.12),
    fontsize=7,
    frameon=False,
    ncol=min(4, len(labels)),
)
fig_bc.suptitle(
    "Zero-shot at best component-R² layer (figure 4 models)",
    y=1.18, fontsize=11,
)
plt.tight_layout()